In [ ]:
import itertools
import matplotlib.pyplot as plt
import math
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import time

from sklearn.preprocessing import StandardScaler

In [ ]:
# Display figures inside the notebook
%matplotlib inline

# Optional formatting for plots
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (16, 9)
plt.rcParams['font.size'] = 18
plt.rcParams['lines.markersize'] = 10

# 4. Model selection and regularization

## 4.1 Subset selection

### 4.1.1 Best subset selection

Here we apply the best subset selection approach to the `Hitters` data. We wish to predict a baseball player's `Salary` on the basis of various statistics associated with performance in the previous year.

First of all, we note that the `Salary` variable is missing for some of the players. the `.isna()` method can be used on the data frame to identify the missing observations. It returns an array of the same shape ($n\times p$) as the input data set, with a `True` for any elements that are missing, and a `False` for any non-missing elements. The `.sum()` method can then be used to aggregate the missing-values matrix into a vector ($n\times 1$) of missing value counts at the level of the variables. The `.sum()` method can be applied again to calculate a sum (scalar) of the missing value counts across all variables.

In [ ]:
hitters = pd.read_csv(os.path.join('..', 'data', 'Hitters.csv'))
hitters = hitters.rename(columns={'Unnamed: 0': 'Name'}).set_index('Name')

ℹ️The `.head()` methods displays only the first few observations in a data frame, which is often enough to get an idea. We will be using it to a avoid unnecessarily cluttering up the screen when viewing data frames in a notebook.

In [ ]:
hitters.head()

In [ ]:
hitters.columns

In [ ]:
hitters.shape

In [ ]:
hitters.isna().sum()

In [ ]:
hitters.isna().sum().sum()

Hence we see that `Salary` is missing for 59 players. The `.dropna()` method removes all of the rows that have missing values in any variable. (Note that in previous examples, we have omitted predictors where observations were missing. In this case, we are omitting observations but retaining the predictors. Both are valid ways of dealing with missing values.)

In [ ]:
hitters = hitters.dropna()

In [ ]:
hitters.shape

In [ ]:
hitters.isna().sum().sum()

The `League`, `Division`, and `NewLeague` variables are categorical, so we use `pandas` to generate dummy variables for them. Because we only need $k-1$ dummies to represent $k$ categorical variable values, we use the `drop_first` argument.

In [ ]:
hitters[['League', 'Division', 'NewLeague']]

In [ ]:
dummies = pd.get_dummies(hitters[['League', 'Division', 'NewLeague']], drop_first=True)

In [ ]:
dummies

In [ ]:
hitters = pd.concat([hitters, dummies], axis=1)
hitters.head()

In [ ]:
hitters.drop(['HmRun', 'Runs', 'RBI', 'CRuns'], axis=1, inplace=True)

In [ ]:
hitters.drop(['League', 'Division', 'NewLeague'], axis=1, inplace=True)

For brevity, we assign shortcuts for our predictor and response data sets.

In [ ]:
y = hitters['Salary']
X = hitters.drop('Salary', axis=1)

We perform best subset selection by identifying the best model that contains a given number of predictors, where *best* is quantified using RSS. We define a function to do this. 

Feature = Predictors = Independent variable

In [ ]:
def process_feature_set(y, X, feature_set):
    """Fit model on `feature_set` and calculate RSS"""
    
    mdl = sm.OLS(y, X[list(feature_set)])
    res = mdl.fit()
    RSS = ((res.predict(X[list(feature_set)]) - y) ** 2).sum()

    return {"model": res, "RSS": RSS}

To use *best subset selection* to find models of up to 10 predictors, we iterate over $[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]$ and run the `select_best()` function in each iteration, storing it in a new data frame `models_best`.

To track execution time, we use `time.time`.

$\mathcal{M}_0, \dots, \mathcal{M}_p$

In [ ]:
def select_best(y, X, k):
    """Iterate over all possible feature sets and call `process_feature_set` on each iteration."""
    
    tic = time.time()

    results = []
    for feature_set in itertools.combinations(X.columns, k):
        results.append(process_feature_set(y, X, feature_set))
    
    models = pd.DataFrame(results)
    best_model = models.loc[models['RSS'].argmin()]
    
    toc = time.time()
    print("Processed", models.shape[0], "models on", k, "predictors in", (toc-tic), "seconds.")
    
    return best_model

In [ ]:
n_candidate_models = 0

for k in range(1, X.shape[1]):
    n_candidate_models += math.comb(X.shape[1], k)

n_candidate_models

In [ ]:
2**19-1

As you can see, there are over half a million model specifications for $k \in [1, \dots, 19]$. We limit ourselves to a maximum number of 10 predictors to ensure (relatively) speedy execution.

ℹ️Because $p \choose k$ candidate models are estimated at every iteration, this will still take a while to run; even limiting ourselves to at most 10 predictors.

In [ ]:
max_k = 10

models_best = pd.DataFrame(columns=['RSS', 'model'])

tic = time.time()
for k in range(1, max_k+1):
    models_best.loc[k] = select_best(y, X, k)
toc = time.time()

print("Total elapsed time:", (toc-tic), "seconds.")

We can inspect the `models_best` data frame to get more information on the models selected as *best* for each $k$.

$\mathcal{M}_0, \dots, \mathcal{M}_p$

In [ ]:
models_best

In [ ]:
print(models_best.loc[2, 'model'].summary())

The best $k=2$-variable model contains only the predictors `Hits` and `CRBI`. To save time, we only generated results up to the best $k=10$-variable model. We can use the functions we have defined to explore feature sets of any size.

In [ ]:
print(select_best(y, X, 15)['model'].summary())

Of course, there is only one model for which $k=p=15$, which is trivially also the best $15$-variable model.

In [ ]:
models_best.head()

In [ ]:
models_best.apply(lambda row: row[1].rsquared, axis=1)

Plotting RSS, adjusted $R^2$, AIC, and BIC for all of the models at once will help us decide which model to select.

In [ ]:
models_best.apply(lambda row: row[1].rsquared_adj, axis=1)

In [ ]:
models_best.apply(lambda row: row[1].aic, axis=1)

In [ ]:
plt.figure()

plt.subplot(2, 2, 1)
plt.plot(models_best['RSS'])
plt.xlabel('# Predictors')
plt.ylabel('RSS')

rsquared_adj = models_best.apply(lambda row: row[1].rsquared_adj, axis=1)

plt.subplot(2, 2, 2)
plt.plot(rsquared_adj)
plt.plot(rsquared_adj.argmax()+1, rsquared_adj.max(), 'or')
plt.xlabel('# Predictors')
plt.ylabel('Adjusted $R^2$')

aic = models_best.apply(lambda row: row[1].aic, axis=1)

plt.subplot(2, 2, 3)
plt.plot(aic)
plt.plot(aic.argmin()+1, aic.min(), 'or')
plt.xlabel('# Predictors')
plt.ylabel('AIC')

bic = models_best.apply(lambda row: row[1].bic, axis=1)

plt.subplot(2, 2, 4)
plt.plot(bic)
plt.plot(bic.argmin()+1, bic.min(), 'or')
plt.xlabel('# Predictors')
plt.ylabel('BIC')

# ❓Question I
1. Which information criterion should you use if you are concerned about obtaining a sparse model (one with a small number of predictors)?
2. The RSS decreases monotonically as a function of the number of predictors in the model. The $R^2$ increases monotonically as a function of the number of predictors in the model. Do you expect the same monotonic behavior for the adjusted $R^2$, if you were to generate the above chart for all possible values of `k`?

### 4.1.2 Stepwise subset selection

Again, we define a function to help us perform subset selection. We begin with *forward stepwise selection*.

In [ ]:
def forward(y, X, predictors):
    
    remaining_predictors = [p for p in X.columns if p not in predictors]
    
    tic = time.time()
    
    results = []
    
    for p in remaining_predictors:
        results.append(process_feature_set(y, X, predictors+[p]))
        
    models = pd.DataFrame(results)
    
    best_model = models.loc[models['RSS'].argmin()]
    
    toc = time.time()
    
    print(f"Processed {models.shape[0]} models on {len(predictors)+1} predictors in {(toc-tic)} seconds.")
    
    return best_model

Independent variables = predictors = exogenous variables = inputs variables = feature

Dependent variables = responses = endogenous variables = output variables

In [ ]:
models = pd.DataFrame(columns=['RSS', 'model'])

tic = time.time()
predictors = []

for k in range(1, len(X.columns)+1):
    models.loc[k] = forward(y, X, predictors)
    predictors = models.loc[k]['model'].model.exog_names
    
toc = time.time()

print(f"Total elapsed time: {(toc-tic)} seconds.")

Backward stepwise selection proceeds analogously.

# ❓Question II
1. We can see that forward stepwise selection was *much* faster than best subset selection. Should we simply always use it instead of best subset selection?

---

# 4.2 Shrinkage/regularization

In [ ]:
from sklearn.preprocessing import scale
from sklearn import model_selection
from sklearn.linear_model import Ridge, RidgeCV, Lasso, LassoCV
from sklearn.metrics import mean_squared_error

Unlike subset selection, which excludes some predictors before fitting a model using least squares, shrinkage methods allow us to fit a model to the entire dataset while mitigating the problems arising with a large $p$. The methods we encountered for this are the *ridge* and the *lasso* estimators (which are available as `Ridge` and `Lasso` in the `sklearn` library).

Because these methods require us to specify a shrinkage parameter $\lambda$, we must use them inconjunction with cross-validation, for which `sklearn` provides `LassoCV` and `RidgeCV`.

We will apply these methods to the `Hitters` data set. We will re-import it for this purpose.

In [ ]:
hitters = pd.read_csv(os.path.join('..', 'data', 'Hitters.csv')).drop('Unnamed: 0', axis=1).dropna()

In [ ]:
dummies = pd.get_dummies(hitters[['League', 'Division', 'NewLeague']])

In [ ]:
y = hitters['Salary']
X_ = hitters.drop(['Salary', 'League', 'Division', 'NewLeague'], axis=1)
X = pd.concat([X_, dummies[['League_N', 'Division_W', 'NewLeague_N']]], axis=1)

In [ ]:
X.isna().sum().sum()

In [ ]:
X.head()

### 4.2.1 Ridge regression

# $\alpha = \lambda$

Because the $\lambda$ shrinkage parameter must be chosen exogenously, we generate a list of equally spaces values for $\lambda$ to find out which value performs best.

In [ ]:
alphas = 10**np.linspace(10, -2, 100)*0.5
print(alphas)

Each value of $\lambda$ will lead to a different set of ridge regression coefficients, which we will store in a matrix called `coefs`. In this case, it is a $19\times 100$ matrix, with 19 rows (one for each predictor) and 100 columns (one for each value of alpha).

Because ridge estimates are not what is called scale-equivariant so that scale matters, we use a Ridge model that normalizes the data set. This can be achieved simply by passing the `normalize` argument to `Ridge`.

In [ ]:
ridge = Ridge()

In [ ]:
coefs = []

for a in alphas:
    ridge.set_params(alpha=a)
    ridge.fit(StandardScaler().fit_transform(X), y)
    coefs.append(ridge.coef_)

In [ ]:
np.shape(coefs)

For larger $\lambda$, we expect the coefficients to be smaller, because ridge regression biases them towards zero. Let's check visually whether this is the case.

In [ ]:
plt.plot(alphas, coefs)
plt.xscale('log')
plt.xlabel('$\lambda$')
plt.ylabel('coefficients')
plt.legend(X.columns)

#### Finding the optimal value of $\lambda$

Now we split the sample into a training set and a test set to find out which value of $\lambda$ leads to the lowest test error.

In [ ]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.5, random_state=1)

We then fit a ridge regression on the training set and evaluate its test MSE for $\lambda=4$.

In [ ]:
ridge2 = Ridge(alpha=4)
ridge2.fit(StandardScaler().fit_transform(X_train), y_train)

pred2 = ridge2.predict(X_test)

In [ ]:
print(pd.Series(ridge2.coef_, index=X.columns))

In [ ]:
print(mean_squared_error(y_test, pred2))

Let's repeat this exercise with a much larger $\lambda = 10^{10}$.

In [ ]:
ridge3 = Ridge(alpha=10**10)
ridge3.fit(StandardScaler().fit_transform(X_train), y_train)

pred3 = ridge3.predict(X_test)

In [ ]:
print(pd.Series(ridge3.coef_, index=X.columns))

In [ ]:
print(mean_squared_error(y_test, pred3))

The very large shrinkage parameter applies a lot of shrinkage so that coefficient estimates hardly respons to the data.

To find the optimal value of $\lambda$, we use cross validation.

In [ ]:
ridge_cv = RidgeCV(alphas=alphas, scoring='neg_mean_squared_error')
ridge_cv.fit(StandardScaler().fit_transform(X_train), y_train)
ridge_cv.alpha_

This gives us the value of $\lambda$/$\alpha$ that is optimal in a test mean squared error sense. Lets fit a value with this shrinkage parameter and look at the MSE we can achieve with it.

In [ ]:
ridge4 = Ridge(alpha=ridge_cv.alpha_)
ridge4.fit(StandardScaler().fit_transform(X_train), y_train)

mean_squared_error(y_test, ridge4.predict(X_test))

This is an improvement over the shrinkage parameter values we tried manually earlier. We can now fit a model with this optimal parameter to our entire data set.

In [ ]:
ridge4.fit(X, y)
print(pd.Series(ridge4.coef_, index=X.columns))

As expeted, some of the coefficients are close to zero, but none are equal to zero exactly.

### 4.2.2 Lasso regression

We saw that ridge regression, with an optimal choise of the shrinkage parameter can outperform least squares for the `Hitters` data set. We now fit a lasso model to see if we can get a better-performing or more interpretable model yet.

In [ ]:
lasso = Lasso(max_iter=10000)

coefs = []

for a in alphas:
    
    lasso.set_params(alpha=a)
    lasso.fit(StandardScaler().fit_transform(X_train), y_train)
    coefs.append(lasso.coef_)


In [ ]:
plt.plot(alphas, coefs)
plt.xscale('log')
plt.xlabel('lambda, alpha')
plt.legend(X.columns)

Note how some of the coefficients are set to zero exactly. The lasso not only provides shrinkage but also performs variable selection.

Now, to choose the optimal shrinkage parameter $\lambda/\alpha$, we perform 10-fold cross validation and then re-fit the model

In [ ]:
lasso_cv = LassoCV(alphas=None, cv=10, max_iter=100000)
lasso_cv.fit(StandardScaler().fit_transform(X_train), y_train)

lasso.set_params(alpha=lasso_cv.alpha_)
lasso.fit(X_train, y_train)

In [ ]:
mean_squared_error(y_test, lasso.predict(X_test))

This is a substantial improvement over least squares, but slightly worse than the ridge model. However, the lasso has the advantage of performing variable selection. Let's have a look at the coefficient estimates.

In [ ]:
print(pd.Series(lasso.coef_, index=X.columns))

# ❓Question III
1. For the ridge regression, the coefficients converge to zero as $\lambda$/$\alpha$ increases, but they never quite reach zero exactly. For the lasso regression, very small coefficients are set to zero exactly. How does this feature of the two methods relate to the norm involved in the shrinkage penalty?